In [ ]:
from pathlib import Path

# === RUN THIS FIRST: repo-root anchor so paths resolve from any working dir ===
# Walks up until it finds the repo root (the folder containing both ai/ and src/).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "ai").is_dir() and (p / "src").is_dir()), Path.cwd())

DATASETS = ROOT / "ai" / "data" / "datasets"                      # datasets root
RUNS     = ROOT / "ai" / "runs"                                   # training/eval output
DEPLOY   = ROOT / "ai" / "models" / "subsystem1" / "production"   # deployed .pt models

print("ROOT     =", ROOT)
print("DATASETS =", DATASETS)
print("RUNS     =", RUNS)
print("DEPLOY   =", DEPLOY)

In [ ]:
import os
import glob
import shutil
import random
import yaml

def split_yolo_dataset(src_dir, out_dir, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-5, "Ratios must sum to 1.0!"

    # 1. Scan source directory for all images
    all_images = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        all_images.extend(glob.glob(os.path.join(src_dir, "**", "images", ext), recursive=True))

    # Filter out files that do not actually exist on disk
    all_images = [img for img in all_images if os.path.isfile(img)]

    if not all_images:
        print("Error: No images found. Check your src_dir path!")
        return

    # 2. Pair each image with its label file
    data_pairs = []
    for img_path in all_images:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        img_ext = os.path.splitext(img_path)[1]
        parent_dir = os.path.dirname(os.path.dirname(img_path))
        lbl_path = os.path.join(parent_dir, "labels", f"{base_name}.txt")

        data_pairs.append({
            "img": img_path,
            "lbl": lbl_path if os.path.isfile(lbl_path) else None,
            "filename": f"{base_name}{img_ext}",
            "labelname": f"{base_name}.txt"
        })

    # 3. Shuffle and split
    random.seed(42)
    random.shuffle(data_pairs)

    total = len(data_pairs)
    num_train = int(total * train_ratio)
    num_val = int(total * val_ratio)

    splits = {
        "train": data_pairs[:num_train],
        "val":   data_pairs[num_train:num_train + num_val],
        "test":  data_pairs[num_train + num_val:]
    }

    print(f"Total: {total} | Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

    # 4. Create output folder structure
    for split_name in splits:
        os.makedirs(os.path.join(out_dir, split_name, "images"), exist_ok=True)
        os.makedirs(os.path.join(out_dir, split_name, "labels"), exist_ok=True)

    # 5. Copy (not move) files into the output folder
    for split_name, pairs in splits.items():
        img_dest = os.path.join(out_dir, split_name, "images")
        lbl_dest = os.path.join(out_dir, split_name, "labels")

        for p in pairs:
            shutil.copy2(p["img"], os.path.join(img_dest, p["filename"]))

            if p["lbl"]:
                shutil.copy2(p["lbl"], os.path.join(lbl_dest, p["labelname"]))
            else:
                # Write empty label file for background images
                open(os.path.join(lbl_dest, p["labelname"]), 'w').close()

    # 6. Write data.yaml into the output folder
    yaml_content = {
        'path': os.path.abspath(out_dir),
        'train': 'train/images',
        'val':   'val/images',
        'test':  'test/images',
        'names': {0: 'tin'}
    }
    with open(os.path.join(out_dir, 'data.yaml'), 'w') as f:
        yaml.safe_dump(yaml_content, f, default_flow_style=False)

    print(f"\nDone! Split dataset saved to: {os.path.abspath(out_dir)}")
    print(f"Your original dataset at '{src_dir}' was not touched.")


split_yolo_dataset(
    src_dir=str(DATASETS / "tin.yolov8"),
    out_dir=str(DATASETS / "tin.yolov8.split"),
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)